# Installation/Setup
- Imports the pipeline and plotting helpers from the local BE3D checkout
- Assumes DSSP, Clustal Omega, and MUSCLE are already installed locally (see the repo README)
- Pins Plotly's renderer to a single explicit value, so figures don't render once per auto-detected frontend


In [1]:
import os
import sys
import subprocess
import copy
import yaml
import pandas as pd
import numpy as np
import plotly.io as pio
from IPython.display import display, Image, SVG
from ipywidgets import interact, Dropdown
from ipymolstar import PDBeMolstar

# Force a SINGLE Plotly renderer -- when multiple Jupyter/Plotly frontend extensions are
# active (common in VS Code), Plotly's auto-detection can set pio.renderers.default to a
# combined string like "vscode+notebook_connected", so every fig.show() call renders once
# per registered renderer, stacking visible duplicates of every single plot in the
# notebook. Pinning to one explicit renderer avoids that regardless of what got detected.
pio.renderers.default = 'vscode'

BECLUST3D_PATH = '/Users/ymyung/Projects/BEClust3D/src/beclust3d-public'
sys.path.insert(0, BECLUST3D_PATH)
sys.path.insert(0, os.path.join(BECLUST3D_PATH, 'examples'))

# Reload the helper modules before importing from them. Python caches modules in
# sys.modules, so in a long-lived kernel a plain `from be3d_plotly import ...` keeps using
# the copy imported earlier in the session -- editing be3d_plotly.py and re-running this
# cell would still raise ImportError for anything newly added (or silently use the old
# version of a changed function) until the kernel was restarted. Reloading here makes
# re-running this cell enough to pick up helper edits.
import importlib
import be3d_local_helper
import be3d_plotly
importlib.reload(be3d_local_helper)
importlib.reload(be3d_plotly)

from be3d_local_helper import (
    show_svgs, show_images, plot_residue_dot, plot_ppi_vs_noppi_scatter,
    render_molstar, load_molstar_pdb, color_molstar, chain_values_from_df, edit_yaml_widgets,
)
from be3d_plotly import (
    show_side_by_side, show_stacked, show_figure_dropdown, plot_hypothesis_qa, plot_violin_by_processed_muttype, plot_score_scatter,
    plot_dendrogram, plot_meta_dendrogram, plot_lfc_lfc3d_scatter, plot_plddt_rsa_scatter,
    plot_domain_barplot, plot_plddt_dis_barplot, plot_enrichment_test,
    plot_meta_score_scatter, plot_meta_lfc_lfc3d_scatter, plot_meta_plddt_rsa_scatter,
    plot_meta_domain_barplot, plot_meta_plddt_dis_barplot,
    COLOR_POS, COLOR_NEG,
)

# Assumes DSSP, ClustalO, and MUSCLE are already installed locally (see the public repo's
# README for install instructions) -- unlike the Colab notebook, this one never shells out
# to apt-get/wget for setup.

def run_be3d(yaml_path):
    script = os.path.join(BECLUST3D_PATH, 'examples', 'be3d_local.py')
    # Capture + print explicitly rather than letting the child inherit stdout/stderr --
    # a subprocess's inherited file descriptors don't reliably show up in a notebook
    # cell's own output (Jupyter/Colab capture sys.stdout at the Python level, which a
    # child process's raw fd can bypass), so check=True alone can raise CalledProcessError
    # with no visible clue about what actually went wrong inside be3d_local.py.
    result = subprocess.run([sys.executable, script, yaml_path], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        result.check_returncode()

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def run_be3d_if_needed(yaml_path, done_marker):
    if os.path.exists(done_marker):
        print(f'[skip] {done_marker} already exists')
    else:
        run_be3d(yaml_path)


# Settings
- Choose which mode to run: monomer, ppi, or blind_target
- Loads that mode's default YAML config
- Example gene: KBTBD4-HDAC1 (PDB 8VOJ), which has data to support all three modes


In [2]:
# KBTBD4-HDAC1 (8VOJ) supports all three modes: monomer, ppi (via ppi_diff), and blind_target.
MONOMER_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/KBTBD4_chain_B.yaml'
PPI_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/ppi_diff_KBTBD4_HDAC1.yaml'
BLIND_TARGET_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/blind_target_KBTBD4_HDAC1.yaml'

for label, path in [('monomer', MONOMER_YAML), ('ppi (ppi_diff)', PPI_YAML), ('blind_target', BLIND_TARGET_YAML)]:
    cfg = load_yaml(path)
    print(f"--- {label}: mode='{cfg.get('mode')}' ---")
    shown = {k: cfg[k] for k in ('input_gene', 'input_uniprot', 'input_chain', 'output_dir') if k in cfg}
    print(yaml.safe_dump(shown, sort_keys=False))


--- monomer: mode='monomer' ---
input_gene: KBTBD4
input_uniprot: Q9NVX7-2
input_chain: B
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/KBTBD4_chain_B

--- ppi (ppi_diff): mode='ppi_diff' ---
input_gene: KBTBD4, HDAC1
input_uniprot: Q9NVX7-2, Q13547
input_chain: B, C
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1

--- blind_target: mode='blind_target' ---
input_gene: KBTBD4
input_uniprot: Q9NVX7-2
input_chain: B
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/blind_target/KBTBD4_HDAC1



## Select a mode
- Pick one of monomer / ppi / blind_target below
- Only the section(s) matching the selected mode actually run further down; the others print a skip message

In [3]:
import ipywidgets as widgets

MODE_OPTIONS = [
    ('Monomer -- single target gene, no PPI partner', 'monomer'),
    ('PPI -- target gene(s) compared with vs. without a PPI partner (mode: ppi_diff)', 'ppi'),
    ('Blind target -- target has no screen data of its own, scored purely from PPI partner(s)', 'blind_target'),
]

mode_selector = widgets.RadioButtons(
    options=MODE_OPTIONS, description='Mode:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='750px'),
)

MODE = mode_selector.value

def _on_mode_change(change):
    global MODE
    MODE = change['new']
    print(f"Selected mode: '{MODE}' -- re-run the config editor and the cells below for this mode.")

mode_selector.observe(_on_mode_change, names='value')
display(mode_selector)
print(f"Selected mode: '{MODE}' -- re-run the config editor and the cells below for this mode.")


RadioButtons(description='Mode:', layout=Layout(width='750px'), options=(('Monomer -- single target gene, no P…

Selected mode: 'monomer' -- re-run the config editor and the cells below for this mode.


## Edit config for the selected mode
- Widgets for the selected mode's YAML fields, each with its own description
- Edits are written back to the YAML file as soon as you change a field, before the pipeline runs


In [4]:
YAML_BY_MODE = {'monomer': MONOMER_YAML, 'ppi': PPI_YAML, 'blind_target': BLIND_TARGET_YAML}
EDITABLE_KEYS_BY_MODE = {
    'monomer': ['input_gene', 'input_uniprot', 'input_chain', 'screen_dir', 'screens',
                'output_dir', 'user_pdb', 'user_fasta', 'user_dssp',
                'nRandom', 'structure_radius', 'clustering_radius',
                'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
    'ppi': ['input_gene', 'input_uniprot', 'input_chain', 'screen_dir', 'screens',
            'output_dir', 'user_pdb', 'user_fasta', 'user_dssp', 'score_type', 'skip_existing',
            'nRandom', 'structure_radius', 'clustering_radius',
            'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
    'blind_target': ['input_gene', 'input_uniprot', 'input_chain', 'output_dir',
                      'user_pdb', 'user_fasta', 'user_dssp',
                      'function_for_lfc', 'function_for_lfc3d', 'function_for_meta'],
}

print(f"Editing config for mode '{MODE}' ({YAML_BY_MODE[MODE]}) -- "
      "changes below are written back to the yaml file immediately, picked up the next "
      "time a cell further down runs the pipeline. Nested settings (pthr, database, "
      "mutation_category, qa, ...) aren't exposed here -- edit the yaml file directly for those.")
edit_yaml_widgets(YAML_BY_MODE[MODE], EDITABLE_KEYS_BY_MODE[MODE])


Editing config for mode 'monomer' (/Users/ymyung/Projects/BEClust3D/be3d_test/KBTBD4_chain_B.yaml) -- changes below are written back to the yaml file immediately, picked up the next time a cell further down runs the pipeline. Nested settings (pthr, database, mutation_category, qa, ...) aren't exposed here -- edit the yaml file directly for those.


# BE-QA
- Runs the pipeline for the selected mode (skipped if its output already exists), then shows that mode's QA plots in one cell
- monomer: hypothesis-test scatters (Kolmogorov-Smirnov and Mann-Whitney statistic vs. -log10(p)) plus the processed per-guide LFC violin for the chosen screen
- ppi: both tests and the violin, each shown twice per gene (no-PPI leg, then PPI-mode leg) side by side
- blind_target: only the partner's processed-LFC violin (partners skip the hypothesis test, so there is no KS/MW plot to show)
- Screen/gene is a variable at the top of the cell, with the valid options listed beside it -- set it, then re-run that cell


In [ ]:
if MODE == 'monomer':
    monomer_cfg = load_yaml(YAML_BY_MODE['monomer'])
    monomer_dir, monomer_gene, monomer_uniprot = monomer_cfg['output_dir'], monomer_cfg['input_gene'], monomer_cfg['input_uniprot']
    run_be3d_if_needed(YAML_BY_MODE['monomer'], os.path.join(monomer_dir, 'RUN_COMPLETED.txt'))

    monomer_screens = [s.strip().split('.')[0] for s in monomer_cfg['screens'].split(',')]

    # No ipywidgets here (no Dropdown, no interact/interact_manual, no "Run Interact"
    # button). Anything display()'d from inside an interact() callback goes into that
    # widget's own Output area rather than the cell's normal output stream, and Colab's
    # Plotly renderer only reaches the latter -- so those figures either never appeared or
    # flashed once and vanished. Selections below are plain variables instead: edit the
    # value and re-run the cell. This is the same plain top-level display() path that
    # BE-MetaClust3D (monomer) was already using, which is the one that always worked.
    QA_SCREEN = 'YeoKBTBD4HDAC12025-ABE-Screen-PPI'  # options: 'YeoKBTBD4HDAC12025-ABE-Screen-PPI', 'YeoKBTBD4HDAC12025-CBE-Screen-PPI'
    if QA_SCREEN not in monomer_screens:
        print(f'[note] {QA_SCREEN!r} not in this config\'s screens ({monomer_screens}) -- using {monomer_screens[0]!r}')
        QA_SCREEN = monomer_screens[0]

    screen_name = QA_SCREEN
    print(f'Screen: {screen_name}   (available: {monomer_screens})')

    print('QA (KS2, MW test, all screens) and processed LFC distribution by mutation category '
          '(violin, post mutation_priority + per-category filtering):')
    show_side_by_side(
        plot_hypothesis_qa(monomer_dir, test='KolmogorovSmirnov'),
        plot_hypothesis_qa(monomer_dir, test='MannWhitney'),
        plot_violin_by_processed_muttype(monomer_dir, monomer_gene, screen_name),
        width=600, height=400, spacing=0.08
    )

elif MODE == 'ppi':
    ppi_cfg = load_yaml(YAML_BY_MODE['ppi'])
    ppi_root = ppi_cfg['output_dir']
    gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
    ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

    # The QA/violin plots need the per-gene no_ppi and ppi legs to already exist, so run the
    # pipeline first (mirrors what the BE-Clust3D (PPI) cell does further down). Variant yamls
    # go to the scratch dir, not the cloned repo's yaml dir (see Settings cell comment).
    # skip_existing makes this a no-op on re-run, so re-running just to change a selection
    # below costs nothing.
    def run_ppi_diff_pass(score_type):
        variant_cfg = copy.deepcopy(ppi_cfg)
        variant_cfg['score_type'] = score_type
        variant_yaml = os.path.join('/content/BE3D_example', f'_ppi_diff_{score_type}.yaml')
        os.makedirs('/content/BE3D_example', exist_ok=True)
        with open(variant_yaml, 'w') as f:
            yaml.safe_dump(variant_cfg, f)
        run_be3d(variant_yaml)

    run_ppi_diff_pass('LFC3D')

    # No ipywidgets here (no Dropdown, no interact/interact_manual, no "Run Interact"
    # button). Anything display()'d from inside an interact() callback goes into that
    # widget's own Output area rather than the cell's normal output stream, and Colab's
    # Plotly renderer only reaches the latter -- so those figures either never appeared or
    # flashed once and vanished. Selections below are plain variables instead: edit the
    # value and re-run the cell. This is the same plain top-level display() path that
    # BE-MetaClust3D (monomer) was already using, which is the one that always worked.
    QA_GENE = 'KBTBD4'  # options: 'KBTBD4', 'HDAC1'
    if QA_GENE not in gene_names:
        print(f'[note] {QA_GENE!r} not in this config\'s genes ({gene_names}) -- using {gene_names[0]!r}')
        QA_GENE = gene_names[0]
    QA_SCREEN = 'YeoKBTBD4HDAC12025-ABE-Screen-PPI'  # options: 'YeoKBTBD4HDAC12025-ABE-Screen-PPI', 'YeoKBTBD4HDAC12025-CBE-Screen-PPI'
    if QA_SCREEN not in monomer_screens:
        print(f'[note] {QA_SCREEN!r} not in this config\'s screens ({monomer_screens}) -- using {monomer_screens[0]!r}')
        QA_SCREEN = monomer_screens[0]

    gene = QA_GENE
    screen_name = QA_SCREEN
    print(f'Gene: {gene}   (available: {gene_names})')
    print(f'Screen: {screen_name}   (available: {ppi_screens})')

    noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
    ppi_dir = os.path.join(ppi_root, 'ppi', gene)

    print(f'{gene} -- QA (KS2 test), no-PPI then PPI-mode:')
    show_side_by_side(
        plot_hypothesis_qa(noppi_dir, test='KolmogorovSmirnov'),
        plot_hypothesis_qa(ppi_dir, test='KolmogorovSmirnov'),
        width=600, height=400,
    )
    print(f'{gene} -- QA (MW test), no-PPI then PPI-mode:')
    show_side_by_side(
        plot_hypothesis_qa(noppi_dir, test='MannWhitney'),
        plot_hypothesis_qa(ppi_dir, test='MannWhitney'),
        width=600, height=400,
    )
    print('Processed LFC distribution by mutation category (violin, post mutation_priority + '
          'per-category filtering), no-PPI then PPI-mode:')
    show_side_by_side(
        plot_violin_by_processed_muttype(noppi_dir, gene, screen_name),
        plot_violin_by_processed_muttype(ppi_dir, gene, screen_name),
        width=600, height=400,
    )

else:
    blind_cfg = load_yaml(YAML_BY_MODE['blind_target'])
    blind_dir = blind_cfg['output_dir']
    blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
    blind_partners = blind_cfg['partners']
    blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

    # blind_target's target itself never gets hypothesis_test/screendata (run_blind_target skips
    # both -- it has no screen data of its own). Each partner runs through parse_be_data +
    # prioritize_by_sequence (preprocess_ppi_partner), same as a monomer run, but
    # preprocess_ppi_partner explicitly skips hypothesis_test -- so there's no KS2/MW QA to show
    # even for the partner, just its processed-LFC violin, under
    # {blind_dir}/ppi_partners/{gene}_chain_{chain}/screendata/.
    if not os.path.exists(blind_tsv):
        run_be3d(YAML_BY_MODE['blind_target'])

    partner_by_gene = {p['gene']: p for p in blind_partners}
    partner_names = list(partner_by_gene)
    partner_screens_by_gene = {
        gene: [s.strip().split('.')[0] for s in p['screens'].split(',')]
        for gene, p in partner_by_gene.items()
    }

    # No ipywidgets here (no Dropdown, no interact/interact_manual, no "Run Interact"
    # button). Anything display()'d from inside an interact() callback goes into that
    # widget's own Output area rather than the cell's normal output stream, and Colab's
    # Plotly renderer only reaches the latter -- so those figures either never appeared or
    # flashed once and vanished. Selections below are plain variables instead: edit the
    # value and re-run the cell. This is the same plain top-level display() path that
    # BE-MetaClust3D (monomer) was already using, which is the one that always worked.
    QA_PARTNER = 'HDAC1'  # options: 'HDAC1'
    if QA_PARTNER not in partner_names:
        print(f'[note] {QA_PARTNER!r} not in this config\'s partners ({partner_names}) -- using {partner_names[0]!r}')
        QA_PARTNER = partner_names[0]
    QA_SCREEN = 'YeoKBTBD4HDAC12025-ABE-Screen-PPI'  # options: 'YeoKBTBD4HDAC12025-ABE-Screen-PPI', 'YeoKBTBD4HDAC12025-CBE-Screen-PPI'
    if QA_SCREEN not in monomer_screens:
        print(f'[note] {QA_SCREEN!r} not in this config\'s screens ({monomer_screens}) -- using {monomer_screens[0]!r}')
        QA_SCREEN = monomer_screens[0]

    partner_gene = QA_PARTNER
    partner_screens = partner_screens_by_gene[partner_gene]
    screen_name = QA_SCREEN
    print(f'Partner: {partner_gene}   (available: {partner_names})')
    print(f'Screen: {screen_name}   (available: {partner_screens})')

    print('Note: blind_target partners skip hypothesis_test (preprocess_ppi_partner), so no '
          "KS2/MW QA plot is available -- showing the partner's processed LFC distribution "
          '(violin) instead:')
    partner_chain = partner_by_gene[partner_gene]['chain']
    partner_dir = os.path.join(blind_dir, 'ppi_partners', f'{partner_gene}_chain_{partner_chain}')
    violin_fig = plot_violin_by_processed_muttype(partner_dir, partner_gene, screen_name)
    if violin_fig is not None:
        display(violin_fig)


# Monomer Mode


## BE-Clust3D
- Residue dot-plots of LFC and LFC3D (positive and negative shown separately), for the chosen screen
- LFC vs. LFC3D scatter, highlighting residues that have an LFC3D value but no direct LFC value
- pLDDT vs. RSA scatter, LFC3D hit count by protein domain, and hit count by pLDDT-disorder category
- Enrichment test (log2 odds ratio) for pLDDT-disorder category
- Screen/gene is a variable at the top of the cell, with the valid options listed beside it -- set it, then re-run that cell


In [ ]:
if MODE == 'monomer':
    # Screen selection is a dropdown form field, not an ipywidgets Dropdown (see the BE-QA
    # cell for why there are no widgets here) -- pick a value and re-run. Changing it re-reads
    # a different screen's tables, so it does need the re-run. The dendrogram is in its own
    # cell below and switches variants without one.
    CLUST3D_SCREEN = ''  # '' = use the first available option (printed below)

    screen_name = CLUST3D_SCREEN
    print(f'Screen: {screen_name}   (available: {monomer_screens})')

    print('Residue dot-plots, LFC (positive, negative):')
    show_side_by_side(
        plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='positive'),
        plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC', direction='negative'),
        width=600, height=400
    )
    print('Residue dot-plots, LFC3D (positive, negative):')
    show_side_by_side(
        plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='positive'),
        plot_score_scatter(monomer_dir, monomer_gene, screen_name, score_type='LFC3D', direction='negative'),
        width=600, height=400
    )

    print('LFC vs. LFC3D (residues with LFC3D but no LFC shown in the left strip):')
    fig = plot_lfc_lfc3d_scatter(monomer_dir, monomer_gene, screen_name, width=500, height=400)
    if fig is not None:
        display(fig)

    print('pLDDT vs. RSA / LFC3D hit count by domain / pLDDT-disorder category:')
    show_side_by_side(
        plot_plddt_rsa_scatter(monomer_dir, monomer_gene, screen_name),
        plot_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, screen_name),
        plot_plddt_dis_barplot(monomer_dir, monomer_gene, screen_name),
        height=400, width=600
    )

    print('Enrichment test (pLDDT-disorder, log2 odds ratio):')
    fig = plot_enrichment_test(monomer_dir, monomer_gene, screen_name=screen_name, width=400, height=300)
    if fig is not None:
        display(fig)
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-Clust3D (monomer).")


### Dendrogram
- Spatial hierarchical clustering of the significant residues (p<0.05), rendered as a merge tree
- All four score-type / direction variants are in one plot; switch with the dropdown on the plot itself -- no cell re-run needed
- Variants with no significant residues are left out of the dropdown and reported in a note


In [ ]:
if MODE == 'monomer':
    # All variants are built once and handed to show_figure_dropdown(), which puts them in a
    # single figure behind Plotly's own dropdown (layout.updatemenus). Switching is done
    # client-side by toggling trace visibility, so picking a different score type / direction
    # does NOT re-run this cell. Deliberately not an ipywidgets Dropdown: those render into a
    # widget output area that Colab's Plotly renderer never reaches, which is why the earlier
    # show_picker()/interact() versions came up blank or flashed once and vanished.
    DENDRO_SCREEN = 'YeoKBTBD4HDAC12025-ABE-Screen-PPI'  # options: 'YeoKBTBD4HDAC12025-ABE-Screen-PPI', 'YeoKBTBD4HDAC12025-CBE-Screen-PPI'
    if DENDRO_SCREEN not in monomer_screens:
        print(f'[note] {DENDRO_SCREEN!r} not in this config\'s screens ({monomer_screens}) -- using {monomer_screens[0]!r}')
        DENDRO_SCREEN = monomer_screens[0]

    print(f'Dendrograms (p<0.05) -- screen {DENDRO_SCREEN}; use the dropdown on the plot to switch:')
    show_figure_dropdown({
        'LFC positive': plot_dendrogram(monomer_dir, monomer_gene, DENDRO_SCREEN, score_type='LFC', direction='Positive', height=400),
        'LFC negative': plot_dendrogram(monomer_dir, monomer_gene, DENDRO_SCREEN, score_type='LFC', direction='Negative', height=400),
        'LFC3D positive': plot_dendrogram(monomer_dir, monomer_gene, DENDRO_SCREEN, score_type='LFC3D', direction='Positive', height=400),
        'LFC3D negative': plot_dendrogram(monomer_dir, monomer_gene, DENDRO_SCREEN, score_type='LFC3D', direction='Negative', height=400),
    }, description='Dendrogram:')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-Clust3D dendrogram (monomer).")


## BE-MetaClust3D
- Only shown when the gene has multiple screens (otherwise there is nothing to meta-aggregate)
- Same plots as BE-Clust3D above, but computed on the meta-aggregated score across all screens instead of one screen at a time


In [ ]:
if MODE == 'monomer':
    monomer_func_meta = monomer_cfg['function_for_meta']

    if len(monomer_screens) > 1:
        print('Meta residue dot-plots, meta-LFC (positive, negative):')
        show_side_by_side(
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='positive'),
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='negative'),
            width=600, height=400
        )
        print('Meta residue dot-plots, meta-LFC3D (positive, negative):')
        show_side_by_side(
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='positive'),
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='negative'),
            width=600, height=400
        )

        print('meta-LFC vs. meta-LFC3D (residues with meta-LFC3D but no meta-LFC shown in the left strip):')
        fig = plot_meta_lfc_lfc3d_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, width=500, height=400)
        if fig is not None:
            display(fig)

        print('pLDDT vs. RSA / Meta LFC3D hit count by domain / pLDDT-disorder category:')
        show_side_by_side(
            plot_meta_plddt_rsa_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
            plot_meta_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, function_for_meta=monomer_func_meta),
            plot_meta_plddt_dis_barplot(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
            width=600, height=400
        )

        print('Enrichment test (pLDDT-disorder, log2 odds ratio):')
        fig = plot_enrichment_test(monomer_dir, monomer_gene, screen_name=None, width=400, height=300)
        if fig is not None:
            display(fig)
    else:
        print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-MetaClust3D (monomer).")


### Meta dendrogram
- Meta-aggregated equivalent of the dendrogram above
- Switch variant with the dropdown on the plot; no cell re-run needed


In [ ]:
if MODE == 'monomer' and len(monomer_screens) > 1:
    # All variants are built once and handed to show_figure_dropdown(), which puts them in a
    # single figure behind Plotly's own dropdown (layout.updatemenus). Switching is done
    # client-side by toggling trace visibility, so picking a different score type / direction
    # does NOT re-run this cell. Deliberately not an ipywidgets Dropdown: those render into a
    # widget output area that Colab's Plotly renderer never reaches, which is why the earlier
    # show_picker()/interact() versions came up blank or flashed once and vanished.
    print('Meta dendrograms (p<0.05) -- use the dropdown on the plot to switch:')
    show_figure_dropdown({
        'Meta LFC positive': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='Positive', height=400),
        'Meta LFC negative': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='Negative', height=400),
        'Meta LFC3D positive': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='Positive', height=400),
        'Meta LFC3D negative': plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='Negative', height=400),
    }, description='Dendrogram:')
elif MODE == 'monomer':
    print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-MetaClust3D dendrogram (monomer).")


# PPI mode


## BE-Clust3D
- Runs the ppi_diff pipeline (PPI leg + no-PPI leg, then merge); re-running is cheap because completed steps are skipped
- Residue dot-plots of LFC3D (positive and negative), no-PPI leg then PPI-mode leg side by side
- No-PPI LFC3D (x) vs. PPI-mode LFC3D (y) scatter, to see how each residue's score shifts between the two
- LFC vs. LFC3D scatter, pLDDT vs. RSA scatter, and LFC3D hit count by pLDDT-disorder category, each no-PPI then PPI-mode
- Enrichment test (pLDDT-disorder), no-PPI then PPI-mode
- Screen/gene is a variable at the top of the cell, with the valid options listed beside it -- set it, then re-run that cell


In [ ]:
if MODE == 'ppi':
    ppi_cfg = load_yaml(PPI_YAML)
    ppi_root = ppi_cfg['output_dir']
    gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
    chain_list = [c.strip() for c in ppi_cfg['input_chain'].split(',')]
    ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

    # mode: ppi_diff runs the PPI leg (mode: complex) and no-PPI leg (mode: monomer, per gene)
    # once, then merges -- skip_existing makes each pass a no-op for the pipeline legs once the
    # first pass has run them, so re-running this cell just to change a selection below is cheap.
    # score_type controls only the (cheap) merge step: 'LFC3D' produces one merged TSV+PDB set
    # per screen; 'Meta_LFC3D' produces the meta-aggregated one (needed by the BE-MetaClust3D
    # and Merged-results sections below).
    def run_ppi_diff_pass(score_type):
        variant_cfg = copy.deepcopy(ppi_cfg)
        variant_cfg['score_type'] = score_type
        variant_yaml = os.path.join('/content/BE3D_example', f'_ppi_diff_{score_type}.yaml')
        os.makedirs('/content/BE3D_example', exist_ok=True)
        with open(variant_yaml, 'w') as f:
            yaml.safe_dump(variant_cfg, f)
        run_be3d(variant_yaml)

    run_ppi_diff_pass('LFC3D')
    run_ppi_diff_pass('Meta_LFC3D')

    ppi_func_meta = ppi_cfg['function_for_meta']
    ppi_uniprot_by_gene = dict(zip(gene_names, [u.strip() for u in ppi_cfg['input_uniprot'].split(',')]))

    # Dropdown form fields, not ipywidgets (see the BE-QA cell). Changing one re-reads that
    # gene/screen's tables, so re-run the cell. Dendrogram is in its own cell below.
    CLUST3D_GENE = ''  # '' = use the first available option (printed below)
    CLUST3D_SCREEN = ''  # '' = use the first available option (printed below)

    gene = CLUST3D_GENE
    screen_name = CLUST3D_SCREEN
    print(f'Gene: {gene}   (available: {gene_names})')
    print(f'Screen: {screen_name}   (available: {ppi_screens})')

    chain = dict(zip(gene_names, chain_list))[gene]
    noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
    ppi_dir = os.path.join(ppi_root, 'ppi', gene)

    print(f'{gene} (chain {chain}) -- residue dot-plots, LFC3D positive -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_score_scatter(noppi_dir, gene, screen_name, score_type='LFC3D', direction='positive'),
        plot_score_scatter(ppi_dir, gene, screen_name, score_type='LFC3D', direction='positive'),
        width=600, height=400
    )
    print('residue dot-plots, LFC3D negative -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_score_scatter(noppi_dir, gene, screen_name, score_type='LFC3D', direction='negative'),
        plot_score_scatter(ppi_dir, gene, screen_name, score_type='LFC3D', direction='negative'),
        width=600, height=400
    )

    print('no-PPI LFC3D (x) vs. PPI-mode LFC3D (y):')
    df_screen = pd.read_csv(os.path.join(ppi_root, f'ppi_vs_noppi_{screen_name}.tsv'), sep='\t')
    plot_ppi_vs_noppi_scatter(df_screen[df_screen['gene'] == gene], 'LFC3D', width=500, height=400)

    print('LFC vs. LFC3D -- no-PPI, then PPI-mode (residues with LFC3D but no LFC shown in each left strip):')
    show_side_by_side(
        plot_lfc_lfc3d_scatter(noppi_dir, gene, screen_name),
        plot_lfc_lfc3d_scatter(ppi_dir, gene, screen_name),
        width=600, height=400
    )

    print('pLDDT vs. RSA -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_plddt_rsa_scatter(noppi_dir, gene, screen_name),
        plot_plddt_rsa_scatter(ppi_dir, gene, screen_name),
        width=600, height=400
    )

    print('LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_plddt_dis_barplot(noppi_dir, gene, screen_name),
        plot_plddt_dis_barplot(ppi_dir, gene, screen_name),
        width=600, height=400
    )

    print('Enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_enrichment_test(noppi_dir, gene, screen_name=screen_name),
        plot_enrichment_test(ppi_dir, gene, screen_name=screen_name),
        width=600, height=400
    )
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-Clust3D (ppi).")


### Dendrogram
- LFC3D spatial clustering merge tree for the selected gene/screen
- Switch between the no-PPI and PPI-mode legs and positive/negative with the dropdown on the plot; no cell re-run needed


In [ ]:
if MODE == 'ppi':
    # All variants are built once and handed to show_figure_dropdown(), which puts them in a
    # single figure behind Plotly's own dropdown (layout.updatemenus). Switching is done
    # client-side by toggling trace visibility, so picking a different score type / direction
    # does NOT re-run this cell. Deliberately not an ipywidgets Dropdown: those render into a
    # widget output area that Colab's Plotly renderer never reaches, which is why the earlier
    # show_picker()/interact() versions came up blank or flashed once and vanished.
    DENDRO_GENE = 'KBTBD4'  # options: 'KBTBD4', 'HDAC1'
    if DENDRO_GENE not in gene_names:
        print(f'[note] {DENDRO_GENE!r} not in this config\'s genes ({gene_names}) -- using {gene_names[0]!r}')
        DENDRO_GENE = gene_names[0]

    DENDRO_SCREEN = 'YeoKBTBD4HDAC12025-ABE-Screen-PPI'  # options: 'YeoKBTBD4HDAC12025-ABE-Screen-PPI', 'YeoKBTBD4HDAC12025-CBE-Screen-PPI'
    if DENDRO_SCREEN not in ppi_screens:
        print(f'[note] {DENDRO_SCREEN!r} not in this config\'s screens ({ppi_screens}) -- using {ppi_screens[0]!r}')
        DENDRO_SCREEN = ppi_screens[0]

    noppi_dir = os.path.join(ppi_root, 'no_ppi', DENDRO_GENE)
    ppi_dir = os.path.join(ppi_root, 'ppi', DENDRO_GENE)

    print(f'LFC3D dendrograms (p<0.05) -- {DENDRO_GENE}, screen {DENDRO_SCREEN}; use the dropdown on the plot to switch:')
    show_figure_dropdown({
        'no-PPI positive': plot_dendrogram(noppi_dir, DENDRO_GENE, DENDRO_SCREEN, score_type='LFC3D', direction='Positive', height=400),
        'PPI-mode positive': plot_dendrogram(ppi_dir, DENDRO_GENE, DENDRO_SCREEN, score_type='LFC3D', direction='Positive', height=400),
        'no-PPI negative': plot_dendrogram(noppi_dir, DENDRO_GENE, DENDRO_SCREEN, score_type='LFC3D', direction='Negative', height=400),
        'PPI-mode negative': plot_dendrogram(ppi_dir, DENDRO_GENE, DENDRO_SCREEN, score_type='LFC3D', direction='Negative', height=400),
    }, description='Dendrogram:')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-Clust3D dendrogram (ppi).")


## BE-MetaClust3D
- Only shown when the gene has multiple screens
- Same no-PPI vs. PPI-mode comparisons as BE-Clust3D (ppi) above, but on the meta-aggregated meta-LFC3D score


In [ ]:
if MODE == 'ppi' and len(ppi_screens) > 1:
    df_meta = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')

    # Dropdown form field, not ipywidgets (see the BE-QA cell). Changing it re-reads that
    # gene's tables, so re-run the cell. Dendrogram is in its own cell below.
    META_GENE = 'KBTBD4'  # options: 'KBTBD4', 'HDAC1'
    if META_GENE not in gene_names:
        print(f'[note] {META_GENE!r} not in this config\'s genes ({gene_names}) -- using {gene_names[0]!r}')
        META_GENE = gene_names[0]

    gene = META_GENE
    print(f'Gene: {gene}   (available: {gene_names})')

    chain = dict(zip(gene_names, chain_list))[gene]
    noppi_dir = os.path.join(ppi_root, 'no_ppi', gene)
    ppi_dir = os.path.join(ppi_root, 'ppi', gene)

    print(f'{gene} (chain {chain}) -- meta residue dot-plots, meta-LFC3D positive -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_meta_score_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='positive'),
        plot_meta_score_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='positive'),
        width=600, height=400
    )
    print('meta residue dot-plots, meta-LFC3D negative -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_meta_score_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='negative'),
        plot_meta_score_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='negative'),
        width=600, height=400
    )

    print('no-PPI meta-LFC3D (x) vs. PPI-mode meta-LFC3D (y):')
    plot_ppi_vs_noppi_scatter(df_meta[df_meta['gene'] == gene], 'meta-LFC3D', width=500, height=400)

    print('meta-LFC vs. meta-LFC3D -- no-PPI, then PPI-mode (residues with meta-LFC3D but no meta-LFC shown in each left strip):')
    show_side_by_side(
        plot_meta_lfc_lfc3d_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta),
        plot_meta_lfc_lfc3d_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta),
        width=500, height=500
    )

    print('pLDDT vs. RSA -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_meta_plddt_rsa_scatter(noppi_dir, gene, function_for_meta=ppi_func_meta),
        plot_meta_plddt_rsa_scatter(ppi_dir, gene, function_for_meta=ppi_func_meta),
        width=600, height=400
    )

    print('Meta LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_meta_plddt_dis_barplot(noppi_dir, gene, function_for_meta=ppi_func_meta),
        plot_meta_plddt_dis_barplot(ppi_dir, gene, function_for_meta=ppi_func_meta),
        width=600, height=400
    )

    print('Meta enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
    show_side_by_side(
        plot_enrichment_test(noppi_dir, gene, screen_name=None),
        plot_enrichment_test(ppi_dir, gene, screen_name=None),
        width=600, height=400
    )
elif MODE == 'ppi':
    print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-MetaClust3D (ppi).")


### Meta dendrogram
- Meta-aggregated equivalent of the dendrogram above
- Switch variant with the dropdown on the plot; no cell re-run needed


In [ ]:
if MODE == 'ppi' and len(ppi_screens) > 1:
    # All variants are built once and handed to show_figure_dropdown(), which puts them in a
    # single figure behind Plotly's own dropdown (layout.updatemenus). Switching is done
    # client-side by toggling trace visibility, so picking a different score type / direction
    # does NOT re-run this cell. Deliberately not an ipywidgets Dropdown: those render into a
    # widget output area that Colab's Plotly renderer never reaches, which is why the earlier
    # show_picker()/interact() versions came up blank or flashed once and vanished.
    DENDRO_GENE = 'KBTBD4'  # options: 'KBTBD4', 'HDAC1'
    if DENDRO_GENE not in gene_names:
        print(f'[note] {DENDRO_GENE!r} not in this config\'s genes ({gene_names}) -- using {gene_names[0]!r}')
        DENDRO_GENE = gene_names[0]

    noppi_dir = os.path.join(ppi_root, 'no_ppi', DENDRO_GENE)
    ppi_dir = os.path.join(ppi_root, 'ppi', DENDRO_GENE)

    print(f'Meta LFC3D dendrograms (p<0.05) -- {DENDRO_GENE}; use the dropdown on the plot to switch:')
    show_figure_dropdown({
        'no-PPI positive': plot_meta_dendrogram(noppi_dir, DENDRO_GENE, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='Positive', height=400),
        'PPI-mode positive': plot_meta_dendrogram(ppi_dir, DENDRO_GENE, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='Positive', height=400),
        'no-PPI negative': plot_meta_dendrogram(noppi_dir, DENDRO_GENE, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='Negative', height=400),
        'PPI-mode negative': plot_meta_dendrogram(ppi_dir, DENDRO_GENE, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='Negative', height=400),
    }, description='Dendrogram:')
elif MODE == 'ppi':
    print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-MetaClust3D dendrogram (ppi).")


## Merged results (meta-LFC3D)
- Table of the top 10 residues ranked by |delta meta-LFC3D| (PPI-mode minus no-PPI)
- Molstar structure viewer colored by the selected view: no-PPI meta-LFC3D, PPI-mode meta-LFC3D, or their delta (-2 to +2, white at 0)
- The top 10 |delta| residues from the table are highlighted as spheres
- Table and viewer are separate cells -- Molstar's widget otherwise sits on top of and hides the table


In [ ]:
if MODE == 'ppi':
    df_merged = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')
    df_merged_sorted = df_merged.reindex(df_merged['delta_score'].abs().sort_values(ascending=False).index)

    print('Top 10 residues by |delta meta-LFC3D| (PPI - no-PPI):')
    top10 = df_merged_sorted.head(10)
    display(top10[['gene', 'chain', 'unipos', 'unires', 'noppi_score', 'ppi_score', 'delta_score']])
    top10_unipos = top10['unipos'].tolist()
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping PPI merged results.")


In [ ]:
if MODE == 'ppi':
    # Molstar viewer in its own cell: it renders as a fixed-height widget that otherwise
    # overlaps and hides the table printed by the cell above when both share one output area.
    base_pdb = os.path.join(ppi_root, 'ppi', gene_names[0], 'sequence_structure')
    base_pdb = os.path.join(base_pdb, [f for f in os.listdir(base_pdb) if f.endswith('_processed.pdb')][0])

    merged_views = {
        'No-PPI meta-LFC3D': chain_values_from_df(df_merged, 'noppi_score'),
        'PPI-mode meta-LFC3D': chain_values_from_df(df_merged, 'ppi_score'),
        'Delta (PPI - no-PPI) meta-LFC3D': chain_values_from_df(df_merged, 'delta_score'),
    }

    MERGED_VIEW = 'Delta (PPI - no-PPI) meta-LFC3D'  # options: 'No-PPI meta-LFC3D', 'PPI-mode meta-LFC3D', 'Delta (PPI - no-PPI) meta-LFC3D'
    view_name = MERGED_VIEW if MERGED_VIEW in merged_views else list(merged_views)[0]

    print(f'Structure colored by "{view_name}" (spheres = the top 10 |delta| residues from the table above):')
    merged_widget = PDBeMolstar(hide_water=True, height='500px')
    load_molstar_pdb(merged_widget, base_pdb)
    color_molstar(merged_widget, merged_views[view_name], vmax=2.0, highlight_top_n=10)
    display(merged_widget)
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping PPI merged results (structure view).")


# Blind target mode
- Table of residues that received a blind LFC3D value (meta-aggregated column when the target has multiple partner screens, otherwise the single screen's column)
- Residue dot-plot of that same signed blind LFC3D value
- Molstar 3D structure viewer colored by the selected view (Negative / Positive / Overall); blue marks positive values, red marks negative ones
- Table, dot-plot, and viewer are separate cells -- Molstar's widget otherwise sits on top of and hides the table


In [ ]:
if MODE == 'blind_target':
    blind_cfg = load_yaml(YAML_BY_MODE['blind_target'])
    blind_dir = blind_cfg['output_dir']
    blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
    blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

    if not os.path.exists(blind_tsv):
        run_be3d(YAML_BY_MODE['blind_target'])

    df_blind = pd.read_csv(blind_tsv, sep='\t')

    # use the meta-aggregated column if there's more than one partner screen, else the single screen's
    screen_names = [c[:-len('_LFC3D_blind_overall')] for c in df_blind.columns if c.endswith('_LFC3D_blind_overall')]
    if 'Meta_LFC3D_blind_overall' in df_blind.columns:
        neg_col, pos_col, overall_col = 'Meta_LFC3D_blind_neg', 'Meta_LFC3D_blind_pos', 'Meta_LFC3D_blind_overall'
    else:
        screen_name = screen_names[0]
        neg_col, pos_col, overall_col = f'{screen_name}_LFC3D_blind_neg', f'{screen_name}_LFC3D_blind_pos', f'{screen_name}_LFC3D_blind_overall'

    df_blind_hits = df_blind[(df_blind[neg_col] != '-') | (df_blind[pos_col] != '-')]
    print(f'{blind_gene} (chain {blind_chain}) -- {len(df_blind_hits)}/{len(df_blind)} residues have a blind LFC3D value:')
    display(df_blind_hits[['unipos', 'unires', 'chain', neg_col, pos_col, overall_col]])
else:
    print(f"[skipped] mode is '{MODE}', not 'blind_target' -- skipping blind-target results.")


In [ ]:
if MODE == 'blind_target':
    print('Residue dot-plot (signed value: negative or positive column, whichever is set):')
    signed = pd.to_numeric(df_blind[neg_col].replace('-', pd.NA), errors='coerce')
    signed = signed.fillna(pd.to_numeric(df_blind[pos_col].replace('-', pd.NA), errors='coerce'))
    df_blind_signed = df_blind.copy()
    df_blind_signed['_signed_blind_LFC3D'] = signed
    plot_residue_dot(df_blind_signed, '_signed_blind_LFC3D', f'{blind_gene} blind LFC3D')
else:
    print(f"[skipped] mode is '{MODE}', not 'blind_target' -- skipping blind-target dot-plot.")


In [ ]:
if MODE == 'blind_target':
    # Molstar viewer in its own cell -- it otherwise overlaps and hides the table above.
    overall_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_overall.pdb')
    pos_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_pos.pdb')
    neg_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_neg.pdb')
    # run_blind_target always writes all three PDBs together (same coordinates, different
    # B-factors baked in) when user_pdb is set, so which one is loaded as the base structure
    # doesn't matter -- only the color_data changes per selection below.
    blind_base_pdb = overall_pdb if os.path.exists(overall_pdb) else (pos_pdb if os.path.exists(pos_pdb) else neg_pdb)

    def _blind_chain_values(col):
        values = pd.to_numeric(df_blind[col].replace('-', pd.NA), errors='coerce')
        return {blind_chain: {int(p): float(v) for p, v in zip(df_blind['unipos'], values) if pd.notna(v)}}

    blind_views = {
        'Negative': _blind_chain_values(neg_col),
        'Positive': _blind_chain_values(pos_col),
        'Overall': _blind_chain_values(overall_col),
    }

    BLIND_VIEW = 'Overall'  # options: 'Negative', 'Positive', 'Overall'
    view_name = BLIND_VIEW if BLIND_VIEW in blind_views else 'Overall'

    print(f'3D structure colored by "{view_name}" (blue = positive, red = negative):')
    blind_widget = PDBeMolstar(hide_water=True, height='500px')
    load_molstar_pdb(blind_widget, blind_base_pdb)
    color_molstar(blind_widget, blind_views[view_name], vmax=2.0, highlight_top_n=10)
    display(blind_widget)
else:
    print(f"[skipped] mode is '{MODE}', not 'blind_target' -- skipping blind-target results (structure view).")
